In [1]:
from dfbr.utils.files import get_config
from dfbr.data.dataset import BikeDemandDataset, BikeOptTargetsDataset
from dfbr.models.station_targets import BikeStationTargets
from dfbr.models.cost_head import CostHead
from dfbr.models.mlp import MLP
from dfbr.eval.simmulation import Sim, create_station_dict, create_event_df
from dfbr.training.train import train_one_epoch, evaluate
import pandas as pd
import matplotlib.pylab as plt
import seaborn as sns
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch.nn as nn
import pyepo

#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Setup
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Read config
config = get_config("baseline.yaml")

#Create a dictionary of stations
station_dict = create_station_dict(config["paths"]["stations"], config["paths"]["station_dist_miles"], config["sim"]["start_inv_pct"])
#Sort by id to ensure alignment
station_ids = sorted(station_dict.keys())
#Get parameters for shapes of datasets and models
num_stations = len(station_dict)
capacities = [s.capacity for s in station_dict.values()]
max_cap = max(capacities)


In [2]:
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Load Datasets
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Create datasets
train_ds = BikeDemandDataset(
        file = config["paths"]["input"],
        start_date = config["data"]["train_start_date"],
        end_date = config["data"]["train_end_date"],
        target_cols= [str(id) for id in station_ids],
        input_scale_cols= ['mean_temp', 'precip', 'max_gust'],
        input_no_scale_cols=['sin_day_of_week', 'cos_day_of_week', 'sin_month', 'cos_month'],
        capacities=capacities,
        max_cap=max_cap
    )

training_stats = {'mean': train_ds.mean, 'std': train_ds.std, 'y_mean': train_ds.y_mean, 'y_std': train_ds.y_std}

test_ds = BikeDemandDataset(
        file = config["paths"]["input"],
        start_date = config["data"]["test_start_date"],
        end_date = config["data"]["test_end_date"],
        target_cols= [str(id) for id in station_ids],
        input_scale_cols= ['mean_temp', 'precip', 'max_gust'],
        input_no_scale_cols=['sin_day_of_week', 'cos_day_of_week', 'sin_month', 'cos_month'],
        capacities=capacities,
        max_cap=max_cap,
        is_train=False,
        scaling_factor=training_stats
    )

In [3]:
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Solve for optimal values with ground truth data
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
opt_model = BikeStationTargets(num_stations=num_stations, max_cap=max_cap, total_inventory=config["model"]["total_inventory"])
pyepo_train_ds = BikeOptTargetsDataset(opt_model, train_ds.X, train_ds.c.view(-1, num_stations * (max_cap + 1)), train_ds.y)
pyepo_test_ds = BikeOptTargetsDataset(opt_model, test_ds.X, test_ds.c.view(-1, num_stations * (max_cap + 1)), test_ds.y)

#Wrap Data Loaders
train_dl = DataLoader(pyepo_train_ds, batch_size=config["training"]["batch_size"], shuffle=False)
test_dl = DataLoader(pyepo_test_ds, batch_size=config["training"]["batch_size"], shuffle=False)


Set parameter Username
Set parameter LicenseID to value 2774727
Academic license - for non-commercial use only - expires 2027-02-03
Optimizing for optDataset...


100%|██████████| 731/731 [00:26<00:00, 28.02it/s]

Optimizing for optDataset...



100%|██████████| 365/365 [00:13<00:00, 27.08it/s]


In [4]:
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Instantiate Models and loss functions
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Create MLP
input_size = len(train_ds[0][0])
output_size = num_stations
pred_model = MLP(input_size, output_size, config["model"]["hidden_layers"])

#Create cost head
cost_head = CostHead(capacities=capacities, max_cap=max_cap)
full_model = nn.Sequential(pred_model, cost_head)

#Create optimizer
optimizer = Adam(full_model.parameters(), lr=config["training"]["learning_rate"])
spo = pyepo.func.SPOPlus(opt_model, processes=1)

#Training loop
for epoch in range(config["training"]["epochs"]):
    for x, c, w, z, y in train_dl:
        #Forward pass
        yp = pred_model(x)
        cp = cost_head(yp)
        loss = spo(cp, c, w, z)
        #Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    train_mse, train_cost, opt_train_cost = evaluate(pred_model, cost_head, opt_model, train_dl)

    print(f"Epoch {epoch+1} Loss: {loss:.4f} Train MSE: {train_mse:.4f} Train Cost: {train_cost:.4f}, Optimal Train Cost: {opt_train_cost:.4f}")


Num of cores: 1
Epoch 1 Loss: 128.1199 Train MSE: 6748.5244 Train Cost: 90.7004, Optimal Train Cost: 41.49247606019152


KeyboardInterrupt: 